# COGS 108 - Data Checkpoint

## Authors

- Dhruv Patel: Analysis, Software, Data curation, 
- Elthor Olivas Corral: Experimental investigation, Analysis, Writing - original draft
- Syrine Mekni: Project administration, Conceptualization, Background research, Writing - review & editing
- Yen-Hsianf Chiu: Methodology, Visualization

## Research Question

How has the distribution of ChatGPT prompts shifted from primarily task-based and informational uses toward more interpersonal and intrapersonal uses (e.g., emotional reflection and relationship interpretation) between April 2023 and May 2024, as measured by topic modeling and sentiment analysis of public ChatGPT conversation data?

In this study, time (April 2023–May 2024) is treated as the independent variable and prompt type (task-based vs interpersonal/intrapersonal) as the dependent variable, with prompt length and system-generated messages controlled for, and the analysis grounded in prior literature on human–AI interaction and the social use of conversational agents.

## Background and Prior Work

## Hypothesis


We hypothesize that between April 2023 and May 2024, the proportion of interpersonal and intrapersonal ChatGPT prompts will increase relative to task-based and informational prompts. This expectation is based on prior research showing that as conversational AI systems become more capable, familiar, and designed to be pleasant, approachable, and human-like, users increasingly disclose emotions, seek support, and treat them as confidants. Building on evidence that conversational AI use has diversified beyond narrowly task-oriented queries, we expect social and self-reflective interactions with ChatGPT to become more prevalent over time.

## Data

### Data overview

### Dataset: WildChat – 1M ChatGPT Interaction Logs

- **Dataset Name:** WildChat  
- **Dataset Link:** https://huggingface.co/datasets/allenai/WildChat  
- **Number of Observations:** Approximately 1,043,000 conversations (filtered to April 2023–May 2024)  
- **Number of Variables:** Includes conversation_id, timestamp, conversation (nested messages), language, model, toxicity indicators, and redacted PII flags  
- **Key Variables for This Project:**  
  - `conversation`: Full conversation history with speaker roles (“user” or “assistant”) and message content  
  - `timestamp`: Used to filter conversations within the April 2023–May 2024 window  
  - `language`: Primary language of the conversation (analysis focuses on English)  
  - `turn`: Number of conversational turns  
  - `user_messages`: Extracted user prompts  

### Dataset Shortcomings

- **Selection bias:** Only includes conversations from users who opted to share their data publicly, which may differ from the broader ChatGPT user base.  
- **Language bias:** Predominantly English, limiting generalizability to multilingual or non-English contexts.  
- **Lack of demographic information:** No data on age, gender, education, or other user characteristics.  
- **Potential data quality issues:** Some conversations may include spam, experimentation, or non-serious interactions.  
- **Missing context:** No information about users’ motivations or prior interactions with ChatGPT.

This dataset is well-suited for our research question because it provides large-scale, real-world, timestamped conversations across the entire period of interest (April 2023–May 2024), enabling us to track how prompt types evolve over time.

In [1]:
# Run this code every time when you're actively developing modules in .py files.  It's not needed if you aren't making modules
#
## this code is necessary for making sure that any modules we load are updated here 
## when their source code .py files are modified

%load_ext autoreload
%autoreload 2

In [2]:
print("here is my code for uploading raw data")

# Setup code -- this only needs to be run once after cloning the repo!
# this code downloads the data from its source to the `data/00-raw/` directory
# if the data hasn't updated you don't need to do this again!

# if you don't already have these packages (you should!) uncomment this line
# %pip install requests tqdm datasets

import sys
sys.path.append('./modules') # this tells python where to look for modules to import

# For WildChat dataset from Hugging Face, we use the datasets library instead of get_data
# The WildChat dataset is too large to use the standard get_data approach
# Instead, we'll download it using Hugging Face's datasets library in the next section

print("NOTE: WildChat dataset will be loaded using Hugging Face datasets library.")
print("This dataset is large (~several GB) and will be downloaded/cached automatically.")
print("First-time download may take 10-30 minutes depending on connection speed.")
print("\nTo download the dataset, run the cells in the 'WildChat Dataset' section below.")

here is my code for uploading raw data
NOTE: WildChat dataset will be loaded using Hugging Face datasets library.
This dataset is large (~several GB) and will be downloaded/cached automatically.
First-time download may take 10-30 minutes depending on connection speed.

To download the dataset, run the cells in the 'WildChat Dataset' section below.


### WildChat Dataset: Loading, Cleaning, and Wrangling

The WildChat dataset contains over 1 million real-world conversations between users and ChatGPT. For our analysis, we focus on conversations from April 2023 to May 2024 to examine how prompt types have evolved over this 14-month period.

**Key metrics and their meaning:**

- **User prompts:** The text input from users to ChatGPT. These are our primary unit of analysis. Prompts can range from single-word queries to multi-paragraph requests. We will analyze these to categorize them as task-based (e.g., "Write a Python function to sort a list") versus interpersonal/intrapersonal (e.g., "I'm feeling anxious about my job interview tomorrow, can you help me process this?").

- **Timestamp:** Unix timestamp or datetime indicating when the conversation occurred. This is critical for our temporal analysis, allowing us to track changes month-by-month from April 2023 through May 2024.

- **Turn count:** Number of back-and-forth exchanges in a conversation. Higher turn counts may indicate more complex or engaging interactions. Multi-turn conversations might suggest deeper engagement, which could be more common in interpersonal exchanges.

- **Prompt length (words):** Word count of user messages. Research suggests task-based prompts tend to be shorter and more directive ("Translate this to Spanish"), while interpersonal prompts may be longer and more narrative ("I've been struggling with my relationship and would like to talk through some thoughts..."). We calculate this as an approximate word count by splitting on whitespace.

- **Prompt length (characters):** Character count including spaces. This provides an alternative length metric that captures very short responses and punctuation.

- **Language:** ISO language code (e.g., 'en' for English, 'es' for Spanish). We focus primarily on English ('en') conversations for consistency in NLP analysis, as sentiment analysis and topic modeling tools perform best on English text.

**Dataset concerns:**

The dataset presents several challenges that affect our analysis:

1. **Size and computational constraints:** The dataset is extremely large (several GB), containing over 1 million conversations. To make processing manageable, we work with filtered subsets based on our time period (April 2023 - May 2024) and language (English). Even after filtering, the dataset may contain hundreds of thousands of prompts, requiring efficient data structures and potentially sampling for some analyses.

2. **Temporal data quality:** Not all conversations have clean timestamp data. Some entries may have null or malformed timestamps that need to be excluded. Additionally, the temporal distribution is uneven—some months have significantly more data than others, which could reflect changes in data collection methods, ChatGPT popularity, or user behavior patterns.

3. **Language diversity and code-switching:** While we filter for English conversations, some conversations may contain multiple languages or code-switching (alternating between languages). Technical discussions may include code snippets in programming languages, which we need to handle appropriately in our text analysis.

4. **Privacy and content concerns:** The WildChat creators have applied PII (personally identifiable information) redaction, but some personal content may remain. Privacy redaction may affect our ability to analyze certain types of personal or emotional content—ironically, the most interpersonal conversations may have had the most redaction. We also need to be mindful that some conversations may contain sensitive topics that require ethical handling.

5. **Conversation structure variability:** The dataset includes both single-turn conversations (one user prompt, one assistant response) and multi-turn conversations (extended back-and-forth). We need to decide whether to analyze each user prompt independently or consider conversation-level patterns. For our analysis, we treat each user prompt as an independent observation while also tracking which conversation it belongs to.

6. **Ground truth absence:** We have no verified labels for what constitutes "task-based" versus "interpersonal/intrapersonal" prompts. We will need to develop and validate our categorization approach through topic modeling and potentially manual review of samples.

7. **Selection bias:** Users who share their conversations publicly may differ systematically from those who don't. People sharing emotional or personal conversations might be more open about mental health, while those with sensitive work queries might not share at all. This could affect our ability to detect genuine temporal trends versus shifts in who is sharing data.

In [3]:
## YOUR CODE TO LOAD/CLEAN/TIDY/WRANGLE THE DATA GOES HERE
# Import necessary libraries
import pandas as pd
import numpy as np
from datasets import load_dataset
from datetime import datetime
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully!")
print("\nNote: If you get import errors, install required packages:")
print("pip install datasets pandas numpy matplotlib seaborn tqdm")

Libraries imported successfully!

Note: If you get import errors, install required packages:
pip install datasets pandas numpy matplotlib seaborn tqdm


In [4]:
# Load WildChat dataset from Hugging Face
# Note: This will download the dataset if it's not already cached
# The full dataset is large (~several GB), so this may take some time on first run

print("Loading WildChat dataset from Hugging Face...")
print("Note: First-time download may take 10-30 minutes depending on connection speed.")
print("Subsequent runs will use the cached version.\n")

# Load the dataset
dataset = load_dataset("allenai/WildChat", split="train")

print(f"Dataset loaded successfully!")
print(f"Total conversations in full dataset: {len(dataset):,}")

Loading WildChat dataset from Hugging Face...
Note: First-time download may take 10-30 minutes depending on connection speed.
Subsequent runs will use the cached version.

Dataset loaded successfully!
Total conversations in full dataset: 529,428


In [5]:
# Convert to pandas DataFrame for easier manipulation
print("Converting to pandas DataFrame...")
df_raw = dataset.to_pandas()

print(f"\nDataFrame shape: {df_raw.shape}")
print(f"\nColumns available:")
for col in df_raw.columns:
    print(f"  - {col}")


print("\nSaving raw dataset to disk...")

raw_path = "data/00-raw/wildchat_raw.pkl"
df_raw.to_pickle(raw_path)

print(f"Raw dataset saved to: {raw_path}")
    
print("\n" + "="*60)
print("First few rows:")
print("="*60)
df_raw.head()

Converting to pandas DataFrame...

DataFrame shape: (529428, 10)

Columns available:
  - conversation_id
  - model
  - timestamp
  - conversation
  - turn
  - language
  - openai_moderation
  - detoxify_moderation
  - toxic
  - redacted

Saving raw dataset to disk...
Raw dataset saved to: data/00-raw/wildchat_raw.pkl

First few rows:


,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted
0,26c5dc109920789f9199ff9b37acb8c1,gpt-4,2023-04-10 00:01:08+00:00,"[{'content': 'Write a very long, elaborate, descriptive and detailed shooting script, including ...",1,English,"[{'categories': {'harassment': False, 'harassment/threatening': False, 'hate': False, 'hate/thre...","[{'identity_attack': 0.0001022904907586053, 'insult': 0.00025938861654140055, 'obscene': 0.00028...",False,False
1,e87a1aeb9aafa35c00da39ddeb1139a0,gpt-4,2023-04-10 00:01:10+00:00,"[{'content': 'what are you?', 'language': 'English', 'redacted': False, 'role': 'user', 'toxic':...",1,English,"[{'categories': {'harassment': False, 'harassment/threatening': False, 'hate': False, 'hate/thre...","[{'identity_attack': 9.160337503999472e-05, 'insult': 0.003521170699968934, 'obscene': 0.0002952...",False,False
2,c3415a9e401ff379f29fe3ce02e500dc,gpt-4,2023-04-10 00:02:37+00:00,"[{'content': 'Write an engaging and a constructive article for my Morocco travel guide book on ""...",1,English,"[{'categories': {'harassment': False, 'harassment/threatening': False, 'hate': False, 'hate/thre...","[{'identity_attack': 9.50227549765259e-05, 'insult': 0.0006598142208531499, 'obscene': 0.0001398...",False,False
3,ec6578cd9d69130a769ea307b6e7a874,gpt-4,2023-04-10 00:03:07+00:00,[{'content': 'CONSTRAINTS: 1. ~4000 word limit for short term memory. Your short term memory is...,1,English,"[{'categories': {'harassment': False, 'harassment/threatening': False, 'hate': False, 'hate/thre...","[{'identity_attack': 0.00012766904546879232, 'insult': 0.00014193766401149333, 'obscene': 0.0001...",False,False
4,17827951de7dc8b29e8e2baa1a73e875,gpt-3.5-turbo,2023-04-10 00:03:09+00:00,[{'content': 'اكتب لي بحث عن اهميه نظام اتحاد النقل الجوي الدولي بالنسبه لاشخاص والبضائع والتاءم...,3,Arabic,"[{'categories': {'harassment': False, 'harassment/threatening': False, 'hate': False, 'hate/thre...","[{'identity_attack': 0.004652907606214285, 'insult': 0.02234211005270481, 'obscene': 0.023811286...",False,False


In [6]:
# Examine the structure of the data
print("Dataset Info:")
print(df_raw.info())

print("\n" + "="*60)
print("Sample conversation structure:")
print("="*60)

sample_conv = df_raw.iloc[0]['conversation']

# If it's a numpy array, convert to list
if isinstance(sample_conv, np.ndarray):
    sample_conv = sample_conv.tolist()

# If it's a string, parse it
if isinstance(sample_conv, str):
    sample_conv = json.loads(sample_conv)

# Now sample_conv is guaranteed to be a list
print("\nFirst 2 messages from a sample conversation:")
print(json.dumps(sample_conv[:2], indent=2))

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529428 entries, 0 to 529427
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype             
---  ------               --------------   -----             
 0   conversation_id      529428 non-null  object            
 1   model                529428 non-null  object            
 2   timestamp            529428 non-null  datetime64[s, UTC]
 3   conversation         529428 non-null  object            
 4   turn                 529428 non-null  int64             
 5   language             529428 non-null  object            
 6   openai_moderation    529428 non-null  object            
 7   detoxify_moderation  529428 non-null  object            
 8   toxic                529428 non-null  bool              
 9   redacted             529428 non-null  bool              
dtypes: bool(2), datetime64[s, UTC](1), int64(1), object(6)
memory usage: 33.3+ MB
None

Sample conversation structure:

First 2 me

In [7]:
# Extract user prompts from conversations
print("Extracting user prompts from conversations...\n")

def extract_user_prompts(conversation):
    """
    Extract all user messages from a conversation.
    Handles lists, numpy arrays, dict-wrapped messages, and JSON strings.
    """

    # If it's a numpy array, convert to list
    if isinstance(conversation, np.ndarray):
        conversation = conversation.tolist()

    # If it's a dict with a 'messages' key, unwrap it
    if isinstance(conversation, dict) and 'messages' in conversation:
        conversation = conversation['messages']

    # If it's a JSON string, parse it
    if isinstance(conversation, str):
        try:
            conversation = json.loads(conversation)
        except json.JSONDecodeError:
            return []

    # If it's still not a list, we can't extract messages
    if not isinstance(conversation, list):
        return []

    # Extract user messages
    user_prompts = []
    for message in conversation:
        if isinstance(message, dict) and message.get('role') == 'user':
            content = message.get('content', '')
            if isinstance(content, str) and content.strip():
                user_prompts.append(content.strip())

    return user_prompts

# Apply extraction with progress bar
tqdm.pandas(desc="Extracting prompts")
df_filtered['user_prompts'] = df_filtered['conversation'].progress_apply(extract_user_prompts)

# Count number of user prompts per conversation
df_filtered['num_user_prompts'] = df_filtered['user_prompts'].apply(len)

print(f"\nExtraction complete!")
print(f"Conversations with at least one user prompt: {(df_filtered['num_user_prompts'] > 0).sum():,}")
print(f"Total user prompts extracted: {df_filtered['num_user_prompts'].sum():,}")
print(f"Average prompts per conversation: {df_filtered['num_user_prompts'].mean():.2f}")

Extracting user prompts from conversations...



NameError: name 'df_filtered' is not defined

In [ ]:
# Create a prompt-level dataset (one row per prompt)
print("Creating prompt-level dataset...\n")

# Explode the user_prompts list so each prompt gets its own row
df_prompts = df_filtered.explode('user_prompts').reset_index(drop=True)

# Remove rows with no prompt
df_prompts = df_prompts[df_prompts['user_prompts'].notna() & (df_prompts['user_prompts'] != '')].copy()

# Rename for clarity
df_prompts.rename(columns={'user_prompts': 'prompt'}, inplace=True)

print(f"Prompt-level dataset shape: {df_prompts.shape}")
print(f"Total prompts: {len(df_prompts):,}")

print("\nSample prompts:")
print("="*60)
for i, prompt in enumerate(df_prompts['prompt'].head(3), 1):
    print(f"{i}. {prompt[:100]}..." if len(prompt) > 100 else f"{i}. {prompt}")
    print()

In [ ]:
# Calculate prompt characteristics
print("Calculating prompt characteristics...\n")

# Prompt length in characters
df_prompts['prompt_length_chars'] = df_prompts['prompt'].str.len()

# Prompt length in words (approximate)
df_prompts['prompt_length_words'] = df_prompts['prompt'].str.split().str.len()

# Extract month and year for temporal analysis
df_prompts['year_month'] = df_prompts['datetime'].dt.to_period('M')
df_prompts['year'] = df_prompts['datetime'].dt.year
df_prompts['month'] = df_prompts['datetime'].dt.month

print("Prompt characteristics added:")
print("  ✓ prompt_length_chars")
print("  ✓ prompt_length_words")
print("  ✓ year_month")
print("  ✓ year")
print("  ✓ month")

print("\nPrompt length summary:")
print(df_prompts[['prompt_length_chars', 'prompt_length_words']].describe())

In [ ]:
# Check for missing data
print("Checking for missing data...\n")
print("Missing values by column:")
print("="*60)

missing_summary = pd.DataFrame({
    'Missing Count': df_prompts.isnull().sum(),
    'Percentage': (df_prompts.isnull().sum() / len(df_prompts) * 100).round(2)
})

missing_data = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_data) > 0:
    print(missing_data)
    
    # Visualize missingness
    plt.figure(figsize=(10, 6))
    plt.barh(missing_data.index, missing_data['Missing Count'])
    plt.xlabel('Number of Missing Values')
    plt.title('Missing Data by Column')
    plt.tight_layout()
    plt.show()
else:
    print("✓ No missing values found in any columns!")

In [ ]:
# Filter to English language conversations (primary focus)
print("Filtering to English language conversations...\n")

if 'language' in df_prompts.columns:
    print("Language distribution (top 10):")
    print("="*60)
    lang_dist = df_prompts['language'].value_counts().head(10)
    for lang, count in lang_dist.items():
        pct = count / len(df_prompts) * 100
        print(f"  {lang}: {count:,} ({pct:.1f}%)")
    
    # Keep only English
    df_prompts_en = df_prompts[df_prompts['language'] == 'English'].copy()
    
    print(f"\nPrompts after English filter: {len(df_prompts_en):,}")
    print(f"Percentage retained: {len(df_prompts_en)/len(df_prompts)*100:.1f}%")
else:
    print("Warning: No language column found. Proceeding with all conversations.")
    df_prompts_en = df_prompts.copy()

In [ ]:
# Identify and handle outliers in prompt length
print("Identifying outliers in prompt length...\n")

# Summary statistics for prompt length
print("Prompt length statistics (words):")
print("="*60)
print(df_prompts_en['prompt_length_words'].describe())

# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_prompts_en['prompt_length_words'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Prompt Length (words)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prompt Lengths')
axes[0].set_xlim(0, 200)  # Focus on reasonable range
axes[0].axvline(df_prompts_en['prompt_length_words'].median(), color='red', linestyle='--', label='Median')
axes[0].legend()

# Box plot
axes[1].boxplot(df_prompts_en['prompt_length_words'], vert=True)
axes[1].set_ylabel('Prompt Length (words)')
axes[1].set_title('Prompt Length Box Plot')
axes[1].set_xticklabels(['Prompts'])

plt.tight_layout()
plt.show()

# Identify extreme outliers (using IQR method)
Q1 = df_prompts_en['prompt_length_words'].quantile(0.25)
Q3 = df_prompts_en['prompt_length_words'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR  # Using 3*IQR for extreme outliers
upper_bound = Q3 + 3 * IQR

outliers = df_prompts_en[
    (df_prompts_en['prompt_length_words'] < lower_bound) | 
    (df_prompts_en['prompt_length_words'] > upper_bound)
]

print(f"\nExtreme outliers detected: {len(outliers):,} ({len(outliers)/len(df_prompts_en)*100:.2f}%)")
print(f"Lower bound: {lower_bound:.1f} words")
print(f"Upper bound: {upper_bound:.1f} words")

In [ ]:
# Clean the data
print("Cleaning data...\n")

initial_count = len(df_prompts_en)

# Remove extremely short prompts (likely noise, testing, or errors)
# We'll keep prompts with at least 2 words
df_clean = df_prompts_en[df_prompts_en['prompt_length_words'] >= 2].copy()
removed_short = initial_count - len(df_clean)
print(f"✓ Removed {removed_short:,} prompts with < 2 words")

# Remove extremely long prompts (likely copy-paste of documents, spam, or errors)
# We'll cap at 99th percentile
length_threshold = df_clean['prompt_length_words'].quantile(0.99)
before_long = len(df_clean)
df_clean = df_clean[df_clean['prompt_length_words'] <= length_threshold].copy()
removed_long = before_long - len(df_clean)
print(f"✓ Removed {removed_long:,} prompts longer than {length_threshold:.0f} words (99th percentile)")

# Remove any rows with missing critical fields
critical_columns = ['prompt', 'datetime', 'year_month']
before_missing = len(df_clean)
df_clean = df_clean.dropna(subset=critical_columns)
removed_missing = before_missing - len(df_clean)
print(f"✓ Removed {removed_missing:,} rows with missing critical data")

# Remove duplicate prompts from the same conversation ID on the same day
# (Keep first occurrence)
if 'conversation_id' in df_clean.columns:
    before_dedup = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=['conversation_id', 'prompt', 'datetime'], keep='first')
    removed_dupes = before_dedup - len(df_clean)
    print(f"✓ Removed {removed_dupes:,} duplicate prompts")

print(f"\n" + "="*60)
print(f"CLEANING SUMMARY")
print("="*60)
print(f"Initial prompts: {initial_count:,}")
print(f"Final cleaned prompts: {len(df_clean):,}")
print(f"Total removed: {initial_count - len(df_clean):,}")
print(f"Retention rate: {len(df_clean)/initial_count*100:.1f}%")

In [ ]:
# Verify data cleanliness with comprehensive visualizations
print("Verifying data cleanliness...\n")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Temporal distribution
temporal_dist = df_clean.groupby('year_month').size()
temporal_dist.plot(kind='bar', ax=axes[0, 0], color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Number of Prompts')
axes[0, 0].set_title('Prompts per Month (Apr 2023 - May 2024)', fontsize=12, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Prompt length distribution (cleaned)
axes[0, 1].hist(df_clean['prompt_length_words'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].axvline(df_clean['prompt_length_words'].mean(), color='red', linestyle='--', 
                   label=f"Mean: {df_clean['prompt_length_words'].mean():.1f}")
axes[0, 1].axvline(df_clean['prompt_length_words'].median(), color='orange', linestyle='--', 
                   label=f"Median: {df_clean['prompt_length_words'].median():.1f}")
axes[0, 1].set_xlabel('Prompt Length (words)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Cleaned Prompt Length Distribution', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Prompt length over time
monthly_length = df_clean.groupby('year_month')['prompt_length_words'].agg(['mean', 'std'])
monthly_length['mean'].plot(ax=axes[1, 0], marker='o', color='coral', linewidth=2, markersize=6)
axes[1, 0].fill_between(monthly_length.index.astype(str), 
                        monthly_length['mean'] - monthly_length['std'],
                        monthly_length['mean'] + monthly_length['std'],
                        alpha=0.2, color='coral')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Average Prompt Length (words)')
axes[1, 0].set_title('Average Prompt Length Over Time (±1 SD)', fontsize=12, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(alpha=0.3)

# 4. Data completeness check
completeness = pd.DataFrame({
    'Column': critical_columns,
    'Completeness %': [100 - (df_clean[col].isna().sum() / len(df_clean) * 100) for col in critical_columns]
})
colors = ['green' if x == 100 else 'orange' for x in completeness['Completeness %']]
axes[1, 1].barh(completeness['Column'], completeness['Completeness %'], color=colors, edgecolor='black')
axes[1, 1].set_xlabel('Completeness (%)')
axes[1, 1].set_title('Data Completeness for Critical Columns', fontsize=12, fontweight='bold')
axes[1, 1].set_xlim(95, 100)
axes[1, 1].grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(completeness['Completeness %']):
    axes[1, 1].text(v - 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('results/data_cleanliness_check.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Data cleanliness visualizations saved to: data/02-processed/data_cleanliness_check.png")

In [ ]:
# Final summary statistics
print("="*70)
print("FINAL CLEANED DATASET SUMMARY")
print("="*70)

print(f"\n📊 Dataset Size:")
print(f"  Total prompts: {len(df_clean):,}")
print(f"  Unique conversations: {df_clean['conversation_id'].nunique() if 'conversation_id' in df_clean.columns else 'N/A'}")

print(f"\n📅 Temporal Coverage:")
print(f"  Start date: {df_clean['datetime'].min()}")
print(f"  End date: {df_clean['datetime'].max()}")
print(f"  Duration: {(df_clean['datetime'].max() - df_clean['datetime'].min()).days} days")
print(f"  Months covered: {df_clean['year_month'].nunique()}")

print(f"\n📝 Prompt Length Statistics (words):")
length_stats = df_clean['prompt_length_words'].describe()
for stat in ['mean', 'std', 'min', '25%', '50%', '75%', 'max']:
    print(f"  {stat:>6s}: {length_stats[stat]:>8.1f}")

print(f"\n📈 Monthly Distribution:")
monthly_counts = df_clean.groupby('year_month').size()
print(f"  Average prompts per month: {monthly_counts.mean():.0f}")
print(f"  Min prompts in a month: {monthly_counts.min():,}")
print(f"  Max prompts in a month: {monthly_counts.max():,}")

print(f"\n🔧 Data Types:")
key_cols = ['prompt', 'datetime', 'prompt_length_words', 'year_month']
for col in key_cols:
    print(f"  {col}: {df_clean[col].dtype}")

print("\n" + "="*70)

In [ ]:
# Save the final cleaned dataset
print("Saving cleaned dataset...\n")

# Select relevant columns for analysis
columns_to_save = [
    'conversation_id',
    'prompt',
    'datetime',
    'year_month',
    'year',
    'month',
    'prompt_length_chars',
    'prompt_length_words',
    'language',
    'turn',
]

# Filter to only columns that exist
columns_to_save = [col for col in columns_to_save if col in df_clean.columns]

df_final = df_clean[columns_to_save].copy()

print(f"Selected {len(columns_to_save)} columns for final dataset:")
for col in columns_to_save:
    print(f"  - {col}")

# Save in multiple formats for flexibility
print("\nSaving files...")

# Pickle format (preserves data types)
df_final.to_pickle('data/02-processed/wildchat_prompts_cleaned.pkl')
print("  ✓ Pickle: data/02-processed/wildchat_prompts_cleaned.pkl")

# CSV format (human-readable)
df_final.to_csv('data/02-processed/wildchat_prompts_cleaned.csv', index=False)
print("  ✓ CSV: data/02-processed/wildchat_prompts_cleaned.csv")

# Save a sample for quick inspection
sample_size = min(1000, len(df_final))
df_final.sample(sample_size).to_csv('data/02-processed/wildchat_prompts_sample.csv', index=False)
print(f"  ✓ Sample ({sample_size:,} rows): data/02-processed/wildchat_prompts_sample.csv")

print("\n" + "="*70)
print("✅ DATA WRANGLING COMPLETE!")
print("="*70)
print("\nNext steps:")
print("  1. Topic modeling to categorize prompts")
print("  2. Sentiment analysis")
print("  3. Temporal analysis of prompt type distribution")

## Ethics

As a team, we recognize that data analysis is not neutral. Even descriptive analytics can influence narratives, policy decisions, and future system design. Because our project analyzes real‑world ChatGPT conversations (WildChat) to study shifts in interpersonal and emotional usage over time, we approach our work with careful attention to privacy, fairness, and unintended downstream consequences.

First, we will ensure that all data used in this project is legally and ethically appropriate for academic analysis. We rely on the WildChat dataset, which is publicly released for research; however, users may not have anticipated a detailed academic analysis of their prompts. Conversational data is inherently sensitive because it may contain emotional disclosures, personal concerns, or other vulnerable expressions. We will therefore avoid reproducing highly sensitive prompts verbatim, refrain from any attempt at re‑identification, and report only aggregate trends rather than individual‑level examples.

Second, we explicitly consider unintended uses of our findings. Because our project examines whether interpersonal and self‑reflective uses of ChatGPT are increasing over time, results could potentially be misinterpreted or repurposed. For example, evidence of increased interpersonal or emotional engagement might be used to optimize systems for user dependency, targeted persuasion, or behavioral profiling. We will frame our findings descriptively and cautiously, avoiding normative claims about whether such trends are inherently beneficial or harmful. Our goal is to inform understanding of usage patterns, not to enable exploitation or manipulation.

Third, we attend to fairness across groups. The WildChat dataset likely overrepresents English‑speaking, technologically literate, and publicly sharing users, so our findings may not generalize to all global ChatGPT users. Additionally, our use of topic modeling and sentiment analysis raises methodological concerns because automated text tools may encode linguistic or cultural biases, especially when interpreting emotion across dialects or languages. Where possible, we will examine available metadata (such as language) and discuss how unequal representation or classification bias may influence our results. We will avoid attributing observed differences to inherent group characteristics and instead acknowledge structural and sampling limitations.

Fourth, we recognize that even descriptive findings can shape downstream system design or policy discussions. If our analysis identifies shifts toward interpersonal usage, those findings could influence how conversational AI systems are framed, regulated, or monetized. Because our project examines whether emotional and interpersonal use is increasing over time, we recognize that framing such trends without context could contribute to narratives about user dependency or vulnerability. We will therefore interpret any observed increase cautiously and avoid implying that emotional engagement with AI is inherently beneficial or harmful.

Finally, we commit to transparency and academic integrity. All preprocessing steps, topic modeling choices, sentiment tools, filtering decisions, and limitations will be documented. We will clearly label assumptions, avoid cherry‑picking results, and acknowledge the constraints of the dataset and analytical methods. External tools, including AI systems, will be used in accordance with course guidelines and properly acknowledged. Our objective is not only to produce accurate analysis but also to ensure that the knowledge generated is responsible, context‑aware, and mindful of its broader social implications.

## Team Expectations 


To maintain accountability and fairness within the group, we have established the following expectations:

### Equal Contribution
- Each team member is expected to actively contribute to coding, analysis, writing, and discussion.  
- Work will be distributed clearly to avoid imbalance.

### Communication
- We will communicate regularly through agreed‑upon platforms.  
- Members should respond within a reasonable timeframe (e.g., 24 hours).  
- If someone is unable to complete a task, they will notify the team as early as possible.

### Deadlines
- Internal deadlines will be set ahead of official due dates to allow time for review and revisions.  
- All members are responsible for meeting these deadlines.

### Respect and Professionalism
- Team discussions will remain respectful and constructive.  
- Disagreements will be handled through open discussion and evidence‑based reasoning.

### Accountability
- Each member will review the final submission to ensure they understand and support the work.  
- No one will submit work they have not reviewed.

### Transparency
- All code will be shared through version control (e.g., Git) so contributions are visible and documented.

---

By setting clear expectations, we aim to create a collaborative, organized, and fair working environment that supports both high‑quality work and positive team dynamics.

## Project Timeline Proposal

In [ ]:
#Over the course of the project, our timeline evolved due to challenges with dataset structure, preprocessing complexity, and the need to strengthen our ethics section based on feedback. Below is our updated and revised timeline reflecting these adjustments.
data = {
    "Meeting Date": [
        "1/28", "2/3", "2/4", "2/11", "2/14",
        "2/25", "3/6", "3/13", "3/20"
    ],
    "Meeting Time": [
        "6 PM", "6 PM", "6 PM", "6 PM", "6 PM",
        "6 PM", "6 PM", "6 PM", "Before 11:59 PM"
    ],
    "Completed Before Meeting": [
        "Read COGS 108 expectations; review project requirements",
        "Complete background research; identify datasets",
        "Revise proposal draft based on dataset constraints",
        "Download and inspect WildChat dataset",
        "Complete data cleaning and initial EDA",
        "Implement topic modeling (LDA) and sentiment analysis",
        "Conduct temporal and distributional analysis",
        "Draft Results, Discussion, Ethics, and Limitations",
        "Final revisions; proofread; confirm citations"
    ],
    "Discuss at Meeting": [
        "Finalize communication channels; confirm research question",
        "Evaluate dataset feasibility; expand ethics (unintended uses, fairness)",
        "Submit updated proposal; assign structured roles",
        "Address formatting inconsistencies; design preprocessing pipeline",
        "Review imbalance across prompt types; discuss bias implications",
        "Evaluate model coherence; refine preprocessing strategy",
        "Interpret findings; strengthen limitations and impact discussion",
        "Peer review full draft; revise for clarity",
        "Submit Final Project & Group Project Surveys"
    ]
}


timeline_df = pd.DataFrame(data)

timeline_df

,Meeting Date,Meeting Time,Completed Before Meeting,Discuss at Meeting
0,1/28,6 PM,Read COGS 108 expectations; review project req...,Finalize communication channels; confirm resea...
1,2/3,6 PM,Complete background research; identify datasets,Evaluate dataset feasibility; expand ethics (u...
2,2/4,6 PM,Revise proposal draft based on dataset constra...,Submit updated proposal; assign structured roles
3,2/11,6 PM,Download and inspect WildChat dataset,Address formatting inconsistencies; design pre...
4,2/14,6 PM,Complete data cleaning and initial EDA,Review imbalance across prompt types; discuss ...
5,2/25,6 PM,Implement topic modeling (LDA) and sentiment a...,Evaluate model coherence; refine preprocessing...
6,3/6,6 PM,Conduct temporal and distributional analysis,Interpret findings; strengthen limitations and...
7,3/13,6 PM,"Draft Results, Discussion, Ethics, and Limitat...",Peer review full draft; revise for clarity
8,3/20,Before 11:59 PM,Final revisions; proofread; confirm citations,Submit Final Project & Group Project Surveys
